[![colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ws2026ies/exercises/blob/main/04_Introduction_to_xarray_PCversion.ipynb)

# Xarray and how to explore Earth system data

The aim of this exercise is to:

* Get an introduction to Xarray
* Access the Earth system analysis data from a STAC catalogue (ERA5)
* Find out about the metadata (which attributes and variables are stored in the cube? What is the resolution (lat, long, temporal)?)
* Plot different variables and make first analyses

---

## Introduction

[`Xarray`](https://docs.xarray.dev/en/stable/index.html) is a python module that ease the usage of labeled multi-dimensional (also known as N-dimensional) arrays built on [`NumPy`](https://numpy.org/) (which you are already familiar with). It allows fast and efficient array-computing, and provides functions for visualizations. Therefore it is very well suited to work with the Earth system data.

In Earth system sciences, it is usually necessary to map data points in their spatial and temporal space (e.g. latitude, longitude and time). Many data formats (e.g. `zarr` or `netCDF`) allow data to be stored together with information about the dimensions and coordinates. Xarray provides two fundamental data structures to work with these data formats: DataArrays and Datasets. In the following image you can see a visual example of these structures ([source](https://docs.xarray.dev/en/stable/_images/dataset-diagram.png)):

![](https://docs.xarray.dev/en/stable/_images/dataset-diagram.png)

- A `DataArray` contains a single multi-dimensional variable and its coordinates
- A `Dataset` holds multiple variables and (potentially) share the same coordinates.

Let's look at a practical example:

## Importing modules
We will use data stored in a STAC catalogue (if you want to know more about STAC, please visit https://stacspec.org/en): The Microsoft's Planetary Computer STAC catalogue (if you want to know more about Planetary Computer, please visit https://planetarycomputer.microsoft.com/). For that we need to install the following packages (you can directly run the following cell if you are in a Google Colab Notebook, if you are in your own computer, you can install them via pip or conda [preferably])

In [ ]:
!pip install planetary_computer adlfs zarr

In [ ]:
# import modules
import numpy as np
import planetary_computer
import xarray as xr
import matplotlib.pyplot as plt
import pystac_client

# Set up data access

### Getting the data (lazily) from Planetary Computer

The following function allows you to open ERA5 data from the Planetary Computer STAC catalogue (https://planetarycomputer.microsoft.com/dataset/era5-pds) lazily (data is not loaded in memory until specified!)

The following arguments are used:

- `datetime`: Either "YYYY" (e.g. "1990") or "YYYY-MM" (e.g. "1990-05"). Data cubes are stored monthly, if no month is passed, the function reads all months in the year and concatenates them. Data is available from 1980 to 2020.
- `dataset`: One of the dataset ids of the ERA5 Planetary Computer STAC collection. Available variables:
  - 'surface_air_pressure'
  - 'sea_surface_temperature'
  - 'eastward_wind_at_10_metres'
  - 'air_temperature_at_2_metres'
  - 'eastward_wind_at_100_metres'
  - 'northward_wind_at_10_metres'
  - 'northward_wind_at_100_metres'
  - 'air_pressure_at_mean_sea_level'
  - 'dew_point_temperature_at_2_metres'

In [ ]:
def get_data_from_pc(datetime="2020-01",dataset="air_temperature_at_2_metres"):

  catalog = pystac_client.Client.open("https://planetarycomputer.microsoft.com/api/stac/v1/")

  search = catalog.search(
      collections=["era5-pds"],
      datetime=datetime,
      query={"era5:kind": {"eq": "an"}}
  )

  items = search.item_collection()

  def sign_and_retrieve(item):

    signed_item = planetary_computer.sign(item)
    asset = signed_item.assets[dataset]

    return xr.open_zarr(asset.href, **{k: asset.extra_fields["xarray:open_kwargs"][k] for k in ["consolidated","storage_options"]})

  return xr.concat([sign_and_retrieve(item) for item in items], "time")

## Open a variable
Let's open a variable as a data cube and store it to the variable `ds` (short for dataset). The original files are in `zarr`. The `zarr` format is used to store chunked and compressed, N-dimensional arrays. Other formats you may know that are used to store big data are `netCDF` or `hdf5`.

In [ ]:
ds = get_data_from_pc(datetime = "2018-07",dataset = "air_temperature_at_2_metres")

Calling the variable `ds` shows us a table with information on dimensions, coordinates, data variables and attributes. You can examine the meta data of the entire dataset or even each individual variable by checking the attributes or clicking on the file icon respectively.

In [ ]:
ds

## DataArrays
The "dot" notation allows you to pull out a dataArray from the dataset, e.g.:

In [ ]:
ds.air_temperature_at_2_metres

In the last exercises we used the "dot" notation to apply a method on a variable (e.g., `a.capitalize()` to capitalize the string `a`, or `geospehere.append('atmosphere')` to add the item `atmosphere` to the list `geospehere`).

The `DataArray` has four important attributes:
* `.coords` is the data container for coordinate variables and their ids. You can access the array values of a coordinate as `<your-dataarray>.<coord-id>.values` (e.g. `ds.air_temperature_at_2_metres.lat.values`)
* `.dims` for the names of each axis (e.g. "lat", "lon", "time" or "x", "y", "z")
* `.attrs` contains all metadata stored in a dictionary
* `.values` is a `numpy.ndarray` that contains the values of the data array (this will load the data into memory! It can take a few minutes)

**TODO**: What is the spatial and temporal resolution of the dataset `ds`? Which information can you find in the metadata of the DataArray `ds.air_temperature_at_2_metres`? Which time period is covered for this variable?

In [ ]:
# Type your answer here

### Label- or position-based indexing

Keep in mind that the data cube comprises several gigabytes of data and running global analyses for the entire time period might be computationally extensive at this resolution. You may want to extract a subset of the dataset. Therefore, one could either use label-based indexing `.sel`, position-based indexing by `.isel` ((integer) select) or slicing of the dataArray:

In [ ]:
# Select the air_temperature_at_2_metres array at time step 750 (which is in this case 2018-07-05T04:00:00)
ds.air_temperature_at_2_metres.isel(time=100)

In [ ]:
# Select all data from the from one day
ds.air_temperature_at_2_metres.sel(time='2018-07-05')

In [ ]:
# If a point is not in the dataste, you can select the nearest
ds.air_temperature_at_2_metres.sel(time='2018-06-15', method = 'nearest')

### Slicing
It is possible to slice the dataArray by applying the `slice()` function to it, that takes the start and end value of the corresponding dimension, i.e. `slice(start, end)`, and only consideres the chosen time period. It is also possible to choose on a spatial area.

In [ ]:
ds.air_temperature_at_2_metres.sel(time=slice("2018-07-10", "2018-07-15"))

## Plotting

With the `.plot()` function, it is easy to visualize the selected data. Since allmeta data are stored along the data (with CF Conventions, learn more here: https://cfconventions.org/), the labeling is done automatically without setting title, x- and y-axis or the legend manually.  

In [ ]:
# Plot a map of one time step (since data has to be loaded into memory, this can take a while)
ds.air_temperature_at_2_metres.sel(time='2018-07-01T12:00:00').plot()

**TODO**: Play around with this function and plot maps of different times.

If the data contains unwanted outliers, visualisation could be improved by adding the parameter `robust=True` inside the `plot()` function. This will use the 2nd and 98th percentiles of the data to compute the color limits.

## Analysis I: How does temperature change over one day in Europe?


First of all, you might have noticed that longitude coordinate values are in [0, 360] degress, so we need to transfer them to [-180, 180] degrees first!

In [ ]:
ds_lons_organized = ds.assign_coords(
    lon=((ds.lon + 180) % 360) - 180
).sortby("lon")

Now, the same slicing we applied to time can be applied to the spatial coordinates!

In [ ]:
ds_europe = ds_lons_organized.sel(
    lat=slice(72, 35),
    lon=slice(-12, 35)
)
ds_europe

Now we can select a day and plot it! Inside the `plot` function you can define subplotting by dimensions using the `col` argument. We will use this to plot each timestamp as a separate subplot. `col_wrap` defines how many columns you want in the layout.

In [ ]:
ds_europe.air_temperature_at_2_metres.sel(time = "2018-07-01").plot(col='time', col_wrap=12, robust=True, cmap="inferno")

**TODO:** Reproduce the previous plot in Celsius (don't forget about the attributes (i.e. "units")).

## Analysis II: How does daily mean temperature change over the whole month?

We would like to know how the air temperature at 2 m changes over the course of the selected month. Therefore, we can resample the dataset from hourly to daily means. This can be done easily by applying `.resample(time = "1D")` (to lear more about offset options and their aliases (e.g. daily, monthly, weekly), see https://pandas.pydata.org/pandas-docs/stable/user_guide/timeseries.html#offset-aliases), followed by the `.mean()` function.

In [ ]:
ds_europe_daily = ds_europe.air_temperature_at_2_metres.resample(time = "1D").mean()
ds_europe_daily # See how our coords changed!

Since data is loaded lazily, we can now compute it in parallel and load into memory (this can take a few minutes):

In [ ]:
ds_europe_daily_memory = ds_europe_daily.compute()

Now, let's plot our daily data!

In [ ]:
ds_europe_daily_memory.plot(col='time', col_wrap=10, robust=True, cmap="inferno")

## Additional material
Find additional information on the Xarray documentation page:

- Tutorials: https://tutorial.xarray.dev/overview/xarray-in-45-min.html
- User guide: https://docs.xarray.dev/en/stable/user-guide/index.html
- Terminology: https://docs.xarray.dev/en/stable/user-guide/terminology.html
- Plotting: https://docs.xarray.dev/en/stable/user-guide/plotting.html

## Classroom exercise

  **TODO**: Select another variable, time period (avoid long periods since data can take a while to compute!), and region of your interest. Can you reproduce the analysis for your variable and region?

**TODO** Additional task: Think about the grid cells. When you plot them, they are all equal in size. What happens if we calculate the e.g. the global mean temperature based on this data? How could it be improved?